### Creating RAG Agent Built on ReAct Architecture

In [1]:
import os
import json
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="prism-ml/ternary-bonsai-2-27b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

data_dir = Path("../data")
pdf_files = sorted(data_dir.glob("**/*.pdf"))

print(f"Found {len(pdf_files)} PDF files")

documents = []

for pdf_file in pdf_files:
    try:
        loader = PyPDFLoader(str(pdf_file))
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = pdf_file.name

        documents.extend(pages)
        print(f"Loaded: {pdf_file.name} ({len(pages)} pages)")

    except Exception as e:
        print(f"Failed: {pdf_file.name} | {e}")

print(f"\nTotal pages loaded: {len(documents)}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13868\1771972328.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Found 15 PDF files
Loaded: 71F0B531C82097D671AD3F254963D548.pdf (66 pages)


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/HFUKWK+HelveticaLTStd-Light', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(688, 0, 1425548464592), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 278, 0, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 0, 0, 0, 0, 0, 0, 0, 667, 0, 0, 722, 611, 0, 778, 0, 278, 0, 0, 556, 833, 0, 778, 611, 0, 0, 611, 556, 722, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 611, 556, 611, 556, 278, 611, 556, 222, 222, 0, 222, 833, 556, 556, 611, 0, 333, 500, 278, 556, 500, 722, 500, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FCVYUO+HelveticaLTStd-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(691, 0, 1

Loaded: g3_science_text_01.pdf (31 pages)


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/LUMVUS+HelveticaLTStd-Light', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(3201, 0, 1425548462992), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 278, 0, 0, 556, 556, 556, 556, 556, 556, 556, 556, 556, 0, 0, 0, 0, 0, 0, 0, 667, 0, 722, 0, 611, 556, 778, 722, 0, 0, 0, 556, 833, 722, 778, 611, 0, 667, 611, 556, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 611, 556, 611, 556, 278, 611, 556, 222, 0, 500, 222, 833, 556, 556, 611, 0, 333, 500, 278, 556, 500, 722, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JRNJSW+HelveticaLTStd-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(3204, 0,

...

Loaded: grade-6-science-and-technology.pdf (226 pages)


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/LNKFGQ+HeinemannRomanSpecial-Bd', '/Encoding': IndirectObject(1374, 0, 1425604493392), '/FirstChar': 31, '/FontDescriptor': IndirectObject(1377, 0, 1425604493392), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [670, 275, 361, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 294, 0, 625, 326, 555, 551, 611, 567, 602, 517, 611, 0, 295, 0, 0, 0, 0, 0, 0, 739, 650, 640, 745, 0, 0, 743, 0, 390, 0, 684, 533, 950, 0, 0, 609, 783, 0, 544, 604, 732, 708, 1026, 0, 0, 581, 0, 0, 0, 0, 0, 0, 636, 640, 479, 643, 577, 401, 640, 637, 276, 276, 565, 321, 940, 637, 640, 640, 0, 405, 457, 396, 637, 554, 848, 547, 554]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/LNKFGQ+HeinemannRomanSpecial-Rm', '/Encoding': IndirectObject(1368, 0, 14256044933

Loaded: Pearson-International-Primary-Science-Y1-Textbook-sample.pdf (20 pages)
Loaded: SCIENCE GRADE 1.pdf (129 pages)
Loaded: science-textbook-grade-2.pdf (144 pages)
Failed: science-textbook-grade-3.pdf | Reached maximum number of tokens per stream.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/TUSXKF+HelveticaLTStd-Light', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(15139, 0, 1425547951440), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 278, 0, 0, 556, 556, 556, 556, 556, 0, 556, 556, 556, 0, 0, 0, 0, 0, 0, 0, 667, 667, 722, 722, 611, 556, 0, 722, 0, 0, 0, 556, 0, 0, 778, 611, 0, 667, 611, 556, 0, 0, 889, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 0, 556, 611, 556, 278, 611, 556, 222, 0, 0, 222, 833, 556, 556, 611, 0, 333, 500, 278, 556, 500, 0, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/TUSXKF+HelveticaLTStd-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(15135, 0, 14255

Loaded: Science-Textbook-Grade-4.pdf (252 pages)
Loaded: science-textbook-grade-5.pdf (208 pages)
Loaded: Science_Y7_EY.pdf (17 pages)
Loaded: Teaching Guide 3.pdf (84 pages)
Loaded: y4-states-of-matter-knowledge-organiser.343941923.pdf (1 pages)
Loaded: y8_pc2_0524_science_revision_guide.pdf (39 pages)

Total pages loaded: 1349


In [3]:
for doc in documents[:3]:
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Text preview:", repr(doc.page_content[:400]))
    print("-" * 50)

Source: 71F0B531C82097D671AD3F254963D548.pdf
Page: 0
Text preview: '1 \n \n \n \nYear 7 KS3 Science \nYear Booklet \nIncludes: \n \n\uf0b7 Home-learning Tasks \n\uf0b7 Revision Questions \n\uf0b7 Content Checklists \nName:______________________ \nTutor Group:_______ \nScience Group:_________'
--------------------------------------------------
Source: 71F0B531C82097D671AD3F254963D548.pdf
Page: 1
Text preview: '2 \n \nContents: \nPg: 4 - 9  7B1: Cells \n \nPg: 10 - 14 7C1: Particles and their Behaviour \n \nPg: 15 - 20 7P4: Space \n \nPg: 21 - 26 7B2: Structure and Function of Body Systems \n \nPg: 27 -30 7C2: Elements, atoms and compounds \n \nPg: 31-36 7P1: Forces \n \nPg: 37-42 7B3: Reproduction \n \nPg: 43-47 7C3: Reactions \n \nPg: 48-53 7P2: Sound \n \nPg: 54-58 7C4: Acids and Alkalis \n \nPg 59 - 66  7P3: Light'
--------------------------------------------------
Source: 71F0B531C82097D671AD3F254963D548.pdf
Page: 2
Text preview: '3 \n \nFAQs. \nHow many home-learning tasks will I n

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[0].page_content[:500])
    print("\nMetadata:", chunks[0].metadata)

Total chunks: 1529

Example chunk:
1 
 
 
 
Year 7 KS3 Science 
Year Booklet 
Includes: 
 
 Home-learning Tasks 
 Revision Questions 
 Content Checklists 
Name:______________________ 
Tutor Group:_______ 
Science Group:_________

Metadata: {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-03-10T17:19:23+00:00', 'author': 'Etheridge.r1', 'moddate': '2018-03-10T17:19:23+00:00', 'source': '71F0B531C82097D671AD3F254963D548.pdf', 'total_pages': 66, 'page': 0, 'page_label': '1', 'start_index': 0}


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "faiss_pdf_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS vector store loaded successfully!")

c:\Users\Lenovo\Desktop\Agentic AI\RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2625.15it/s]


FAISS vector store loaded successfully!


# Testing RAG Model

In [7]:
query = "What happens if an organism was removed from a food chain?"

retrieved_docs = vectorstore.similarity_search(
    query,
    k=4
)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])


--- Result 1 ---
Source: g5_science_text_01.pdf
Page: 20
1919
What happens if an organism was removed 
from a food chain?
 
If this was a food chain in an environment, where plants are eaten by 
grasshoppers and the grasshoppers are eaten by frogs and the frogs are 
eaten by snakes.
If frogs were to die because of some diseases caused by some pollution, 
there would be an increase in the amount of grasshoppers feeding on the 
producer or green plants.
This would cause a major problem because grasshoppers would be out of 
control. They would eat plants and the number of plants which are the basis 
of the food chain would severely decrease.
On the other hand there would be an effect on the consumers of frogs which 
are the snakes. They would lose an

--- Result 2 ---
Source: g5_science_text_01.pdf
Page: 20
environment where there are also other predators like birds. They would feed 
on grasshoppers but in such case if an organism primarily eats one type of 
organism which is the food so

# Connecting Fiass vectorstore with LLM

In [8]:
def rag_answer(query:str)-> str:
    #retrieve relevant documents from the vector store
    retrieved_docs=vectorstore.similarity_search(
        query,
        k=4
    )
    # if no documents are retrieved
    if not retrieved_docs:
        return "No relevant information was found for the query."

    #combine the content of the retrieved chunks into context
    context="\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n"
            f"Page: {doc.metadata.get('page', 'Unknown')}\n"
            f"Content: {doc.page_content}"
            for doc in retrieved_docs
        ]
    )
    # Step 4: Create a prompt containing the context and question
    prompt = f"""
You are a helpful document-based question-answering assistant.

Answer the user's question using the retrieved document context.
Do not make up facts that are not supported by the context.
If the context does not contain enough information, say so.
Include the source filename and page number when relevant.

Retrieved Context:
{context}

User Question:
{query}

Answer:
"""
    response = llm.invoke(prompt)

    # Step 6: Return the generated answer
    return response.content

In [9]:
query = "What happens if an organism is removed from a food chain?"

answer = rag_answer(query)

print(answer)



If an organism is removed from a food chain, the other organisms can be affected. For example, if frogs were removed or die, grasshoppers would increase because their predator would be gone. The grasshoppers could overeat plants, causing the number of plants to decrease. Snakes would also be affected because they would lose a food source, and their numbers could decrease.

In environments with several food chains, other animals may also be affected. If an organism mainly depends on one type of food and that food source disappears, it may die off, which can then affect its predators and possibly lead to their extinction as well.

Source: g5_science_text_01.pdf, page 20.
